# Kaggle S6E3 — Customer Churn v3
5-model ensemble: LGB + XGB + CatBoost (×3 seeds each) + LogReg (OHE) + NN (embeddings).  
TE computed inside each CV fold to prevent leakage. Per-model feature tailoring (Deotte approach).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
SEED = 42; N_SPLITS = 5
np.random.seed(SEED)
print('ok')

In [ ]:
import os
if os.path.exists('/kaggle/input'):
    TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
else:
    TRAIN_PATH = 'data/train.csv'
    TEST_PATH  = 'data/test.csv'
    SUB_PATH   = 'data/sample_submission.csv'
    ORIG_PATH  = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
sub   = pd.read_csv(SUB_PATH)
print('train', train.shape, '| test', test.shape, '| orig', orig.shape)

In [ ]:
TARGET='Churn'
CATS=['gender','SeniorCitizen','Partner','Dependents','PhoneService',
    'MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

train_ids=train['id'].copy(); test_ids=test['id'].copy()
train=train.drop(columns=['id']); test=test.drop(columns=['id'])
if 'customerID' in orig.columns: orig=orig.drop(columns=['customerID'])

def map_target(s): return s.map({'Yes':1,'No':0}).astype('int8') if s.dtype==object else s.astype('int8')
train[TARGET]=map_target(train[TARGET]); orig[TARGET]=map_target(orig[TARGET])

for df in [train,test,orig]:
    df['tenure']=pd.to_numeric(df['tenure'],errors='coerce').astype('float32')
    df['MonthlyCharges']=pd.to_numeric(df['MonthlyCharges'],errors='coerce').astype('float32')
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'].astype(str).str.strip().replace('',np.nan),errors='coerce').astype('float32')
    m=df['TotalCharges'].isna()
    df.loc[m,'TotalCharges']=df.loc[m,'tenure']*df.loc[m,'MonthlyCharges']
    for c in CATS:
        if c in df.columns: df[c]=df[c].astype(str).fillna('Missing')

BASE=[c for c in train.columns if c!=TARGET]
print('BASE cols:',len(BASE),'| churn rate:',round(train[TARGET].mean(),4))

In [ ]:
from itertools import combinations

# ── DIGIT features ────────────────────────────────────────────
def add_digit_cols(df, col, multiplier):
    if multiplier == 'direct':
        vals = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors='coerce') * multiplier).round(0).fillna(-1).astype(int)
    max_len = {'tenure':3,'MonthlyCharges':5,'TotalCharges':6}.get(col, vals.astype(str).str.len().max())
    s = vals.astype(str).str.zfill(max_len)
    created = []
    for i in range(max_len):
        n = f'{col}_DIGIT_{i+1}'; df[n] = s.str[i].astype(int); created.append(n)
    return created

DIGIT = []
for df in [train, test]:
    d = add_digit_cols(df, 'tenure', 'direct')
    add_digit_cols(df, 'MonthlyCharges', 100)
    add_digit_cols(df, 'TotalCharges', 1)
    if df is train:
        DIGIT = d + [f'MonthlyCharges_DIGIT_{i+1}' for i in range(5)] + [f'TotalCharges_DIGIT_{i+1}' for i in range(6)]
print(len(DIGIT), 'DIGIT features')

# ── ROUND features ────────────────────────────────────────────
ROUND = []
for col, levels in [('MonthlyCharges',[('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('TotalCharges',  [('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('tenure',        [('1s',0),('10s',-1)])]:
    for suffix, level in levels:
        n = f'{col}_ROUND_{suffix}'; ROUND.append(n)
        for df in [train, test]: df[n] = df[col].round(level).fillna(-1).astype(int)
print(len(ROUND), 'ROUND features')

# ── ORIG dataset stats ────────────────────────────────────────
ORIG_FEATS = []
orig_mean = float(orig[TARGET].mean())
for col in BASE:
    mm = orig.groupby(col)[TARGET].mean()
    cm = orig.groupby(col).size()
    for df in [train, test]:
        df[f'orig_mean_{col}']  = df[col].map(mm).fillna(orig_mean)
        df[f'orig_count_{col}'] = df[col].map(cm).fillna(0)
    ORIG_FEATS += [f'orig_mean_{col}', f'orig_count_{col}']
print(len(ORIG_FEATS), 'ORIG features')

# ── INTERACTION features (string concat, TE inside CV) ────────
INTER = []
for col1, col2 in combinations(BASE, 2):
    n = f'{col1}_{col2}'; INTER.append(n)
    for df in [train, test]: df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
print(len(INTER), 'INTER features')

FEATURES = list(dict.fromkeys(BASE + ORIG_FEATS + INTER + ROUND + DIGIT))
print(len(FEATURES), 'total features')

In [ ]:
import time

# ── Keep raw data for per-fold TE encoding ────────────────────
X_all     = train[FEATURES].copy()
y_all     = train[TARGET].copy()
Xtest_raw = test[FEATURES].copy()
gm        = float(y_all.mean())

NUM_COLS_BASE = ['tenure', 'MonthlyCharges', 'TotalCharges']

def factorize3(tr, val, te):
    codes, _ = pd.factorize(pd.concat([tr, val, te], ignore_index=True).astype(str))
    n1, n2 = len(tr), len(val)
    return (pd.Series(codes[:n1], index=tr.index, dtype='int32'),
            pd.Series(codes[n1:n1+n2], index=val.index, dtype='int32'),
            pd.Series(codes[n1+n2:], index=te.index, dtype='int32'))

def compute_te(Xtr, ytr, Xval, Xte, inter_cols, m=200):
    """LOO-smoothed TE for training rows, smoothed TE for val/test.
    m = smoothing strength (higher = more shrinkage toward global mean).
    """
    Xtr = Xtr.copy(); Xval = Xval.copy(); Xte = Xte.copy()
    gm = float(ytr.mean())
    
    for col in inter_cols:
        # Group stats from training fold
        grp = Xtr[[col]].assign(_y=ytr.values).groupby(col)['_y']
        grp_sum   = grp.sum()
        grp_count = grp.count()
        
        # Training rows: LOO with smoothing
        col_sum   = Xtr[col].map(grp_sum).fillna(0)
        col_count = Xtr[col].map(grp_count).fillna(0)
        Xtr[f'TE_{col}'] = ((col_sum - ytr.values + m * gm) /
                             (col_count - 1 + m)).astype('float32')
        
        # Val/test rows: standard smoothed TE
        grp_mean_smooth = (grp_sum + m * gm) / (grp_count + m)
        Xval[f'TE_{col}'] = Xval[col].map(grp_mean_smooth).fillna(gm).astype('float32')
        Xte[f'TE_{col}']  = Xte[col].map(grp_mean_smooth).fillna(gm).astype('float32')
    
    Xtr  = Xtr.drop(columns=inter_cols)
    Xval = Xval.drop(columns=inter_cols)
    Xte  = Xte.drop(columns=inter_cols)
    return Xtr, Xval, Xte

# ── LightGBM — multiple seeds, TE computed inside each fold ──
LGB_SEEDS = [42, 123, 456]
lgb_params = {
    'objective':'binary','metric':'auc','n_estimators':2000,'learning_rate':0.05,
    'num_leaves':127,'subsample':0.8,'colsample_bytree':0.6,'min_child_samples':20,
    'reg_alpha':0.1,'reg_lambda':1.0,'n_jobs':-1,'verbose':-1
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros(len(X_all),     dtype=np.float64)
pred_lgb = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    print(f'\n=== LightGBM seed={seed} ===')
    oof_s = np.zeros(len(X_all),     dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)
    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
        ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
        Xte = Xtest_raw.copy()
        # LOO-smoothed TE inside fold
        Xtr, Xval, Xte = compute_te(Xtr, ytr, Xval, Xte, INTER, m=200)
        for c in CATS:
            if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        for c in Xtr.select_dtypes('object').columns:
            Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        params = {**lgb_params, 'random_state': seed}
        model = lgb.LGBMClassifier(**params)
        model.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
        oof_s[vali] = model.predict_proba(Xval)[:, 1]
        tp_s += model.predict_proba(Xte)[:, 1] / N_SPLITS
        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'iter={model.best_iteration_}  time={time.time()-fold_start:.0f}s')
    print(f'  Seed {seed} OOF: {roc_auc_score(y_all, oof_s):.5f}')
    oof_lgb  += oof_s / len(LGB_SEEDS)
    pred_lgb += tp_s  / len(LGB_SEEDS)

print(f'\nLightGBM multi-seed OOF AUC: {roc_auc_score(y_all, oof_lgb):.5f}')
np.save('oof_lgb.npy', oof_lgb); np.save('pred_lgb.npy', pred_lgb)

In [ ]:
import xgboost as xgb

# ── XGBoost — multiple seeds, same TE features as LGB ────────
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'max_depth': 7, 'learning_rate': 0.05, 'n_estimators': 2000,
    'subsample': 0.8, 'colsample_bytree': 0.6,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'tree_method': 'hist', 'n_jobs': -1, 'verbosity': 0,
}

oof_xgb  = np.zeros(len(X_all),     dtype=np.float64)
pred_xgb = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    print(f'\n=== XGBoost seed={seed} ===')
    oof_s = np.zeros(len(X_all),     dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)
    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
        ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
        Xte = Xtest_raw.copy()
        # LOO-smoothed TE inside fold
        Xtr, Xval, Xte = compute_te(Xtr, ytr, Xval, Xte, INTER, m=200)
        for c in CATS:
            if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        for c in Xtr.select_dtypes('object').columns:
            Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        params = {**xgb_params, 'random_state': seed}
        model_xgb = xgb.XGBClassifier(**params)
        model_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                       callbacks=[xgb.callback.EarlyStopping(100, data_name='validation_0',
                                                              save_best=True)])
        oof_s[vali] = model_xgb.predict_proba(Xval)[:, 1]
        tp_s += model_xgb.predict_proba(Xte)[:, 1] / N_SPLITS
        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'iter={model_xgb.best_iteration}  time={time.time()-fold_start:.0f}s')
    print(f'  Seed {seed} OOF: {roc_auc_score(y_all, oof_s):.5f}')
    oof_xgb  += oof_s / len(LGB_SEEDS)
    pred_xgb += tp_s  / len(LGB_SEEDS)

print(f'\nXGBoost multi-seed OOF AUC: {roc_auc_score(y_all, oof_xgb):.5f}')
np.save('oof_xgb.npy', oof_xgb); np.save('pred_xgb.npy', pred_xgb)

In [ ]:
from catboost import CatBoostClassifier

# CatBoost uses raw categoricals natively — no TE needed
# Drop interaction columns (strings), keep only BASE + ORIG + DIGIT + ROUND
CAT_FEATURES = [c for c in FEATURES if c not in INTER]
cat_cols_cb  = [c for c in CATS if c in CAT_FEATURES]
cat_idx_cb   = [CAT_FEATURES.index(c) for c in cat_cols_cb]

cb_params = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, eval_metric='AUC', od_type='Iter', od_wait=100,
    verbose=False, thread_count=-1,
)

oof_cat  = np.zeros(len(X_all),     dtype=np.float64)
pred_cat = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    print(f'\n=== CatBoost seed={seed} ===')
    oof_s = np.zeros(len(X_all),     dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)
    cv_start = time.time()
    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr  = X_all[CAT_FEATURES].iloc[tri].copy()
        Xval = X_all[CAT_FEATURES].iloc[vali].copy()
        ytr  = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
        Xte  = Xtest_raw[CAT_FEATURES].copy()
        model_cb = CatBoostClassifier(**{**cb_params, 'random_seed': seed})
        model_cb.fit(Xtr, ytr, cat_features=cat_idx_cb,
                     eval_set=(Xval, yval), use_best_model=True)
        oof_s[vali] = model_cb.predict_proba(Xval)[:, 1]
        tp_s += model_cb.predict_proba(Xte)[:, 1] / N_SPLITS
        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'best_iter={model_cb.best_iteration_}  time={time.time()-fold_start:.0f}s')
    print(f'  Seed {seed} OOF: {roc_auc_score(y_all, oof_s):.5f}  total={time.time()-cv_start:.0f}s')
    oof_cat  += oof_s / len(LGB_SEEDS)
    pred_cat += tp_s  / len(LGB_SEEDS)

print(f'\nCatBoost multi-seed OOF AUC: {roc_auc_score(y_all, oof_cat):.5f}')
np.save('oof_cat.npy', oof_cat); np.save('pred_cat.npy', pred_cat)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# NN uses only 16 raw categoricals (learned embeddings) + 3 scaled numerics
nn_cat_cols = CATS
nn_num_cols = NUM_COLS_BASE

cat_vocabs = {}
for c in nn_cat_cols:
    all_vals = pd.concat([X_all[c], Xtest_raw[c]], ignore_index=True).astype(str)
    cat_vocabs[c] = {v: i+1 for i, v in enumerate(all_vals.unique())}

def encode_cats(df, vocabs):
    return {c: df[c].astype(str).map(vocab).fillna(0).astype('int64')
            for c, vocab in vocabs.items()}

class ChurnMLP(nn.Module):
    def __init__(self, cat_vocabs, num_dim, hidden=[256, 128, 64], dropout=0.4):
        super().__init__()
        self.cat_names = list(cat_vocabs.keys())
        self.embeddings = nn.ModuleDict({
            c: nn.Embedding(len(v)+1, min(50, (len(v)+1)//2))
            for c, v in cat_vocabs.items()
        })
        emb_total = sum(min(50, (len(v)+1)//2) for v in cat_vocabs.values())
        in_dim = emb_total + num_dim
        layers = [nn.BatchNorm1d(in_dim), nn.Dropout(0.2)]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embs = [self.embeddings[c](x_cat[:, i]) for i, c in enumerate(self.cat_names)]
        return self.net(torch.cat(embs + [x_num], dim=1)).squeeze(1)

def make_tensors(Xdf, yvals=None):
    cat_enc = encode_cats(Xdf, cat_vocabs)
    x_cat = torch.tensor(
        np.stack([cat_enc[c].values for c in nn_cat_cols], axis=1), dtype=torch.long
    ).to(device)
    x_num = torch.tensor(Xdf[nn_num_cols].values.astype('float32')).to(device)
    if yvals is not None:
        return x_cat, x_num, torch.tensor(yvals.values.astype('float32')).to(device)
    return x_cat, x_num

oof_nn  = np.zeros(len(X_all),     dtype=np.float32)
pred_nn = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    scaler = StandardScaler()
    Xtr[nn_num_cols]  = scaler.fit_transform(Xtr[nn_num_cols].values.astype('float32'))
    Xval[nn_num_cols] = scaler.transform(Xval[nn_num_cols].values.astype('float32'))
    Xte[nn_num_cols]  = scaler.transform(Xte[nn_num_cols].values.astype('float32'))

    xc_tr, xn_tr, y_tr = make_tensors(Xtr, ytr)
    xc_val, xn_val, _  = make_tensors(Xval, yval)
    xc_te, xn_te       = make_tensors(Xte)

    loader = DataLoader(TensorDataset(xc_tr, xn_tr, y_tr), batch_size=2048, shuffle=True)
    model_nn = ChurnMLP(cat_vocabs, len(nn_num_cols)).to(device)
    optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
    criterion = nn.BCEWithLogitsLoss()

    best_auc_nn = 0; patience_ct = 0; best_state = None
    for epoch in range(150):
        model_nn.train()
        for xc_b, xn_b, y_b in loader:
            optimizer.zero_grad()
            criterion(model_nn(xc_b, xn_b), y_b).backward()
            optimizer.step()
        scheduler.step()
        model_nn.eval()
        with torch.no_grad():
            val_auc = roc_auc_score(yval, torch.sigmoid(model_nn(xc_val, xn_val)).cpu().numpy())
        if val_auc > best_auc_nn:
            best_auc_nn = val_auc; patience_ct = 0
            best_state = {k: v.clone() for k, v in model_nn.state_dict().items()}
        else:
            patience_ct += 1
            if patience_ct >= 15: break

    model_nn.load_state_dict(best_state)
    model_nn.eval()
    with torch.no_grad():
        oof_nn[vali] = torch.sigmoid(model_nn(xc_val, xn_val)).cpu().numpy()
        pred_nn += torch.sigmoid(model_nn(xc_te, xn_te)).cpu().numpy() / N_SPLITS

    print(f'NN Fold {fold} AUC: {roc_auc_score(yval, oof_nn[vali]):.5f}  '
          f'epochs={epoch-patience_ct+1}  time={time.time()-fold_start:.0f}s')

print(f'\nNN OOF AUC: {roc_auc_score(y_all, oof_nn):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_nn.npy', oof_nn); np.save('pred_nn.npy', pred_nn)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

# ── Logistic Regression — OHE categoricals + scaled numerics + TE ──
oof_lr  = np.zeros(len(X_all),     dtype=np.float64)
pred_lr = np.zeros(len(Xtest_raw), dtype=np.float64)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
    ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
    Xte = Xtest_raw.copy()

    # LOO-smoothed TE inside fold
    Xtr_te, Xval_te, Xte_te = compute_te(Xtr, ytr, Xval, Xte, INTER, m=200)

    # Get TE column names
    te_cols = [c for c in Xtr_te.columns if c.startswith('TE_')]

    # Numeric columns (base + ORIG + DIGIT + ROUND + TE)
    lr_num_cols = NUM_COLS_BASE + ORIG_FEATS + DIGIT + ROUND + te_cols

    # OHE categoricals
    ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
    ohe.fit(Xtr_te[CATS].astype(str))
    Xtr_ohe  = ohe.transform(Xtr_te[CATS].astype(str))
    Xval_ohe = ohe.transform(Xval_te[CATS].astype(str))
    Xte_ohe  = ohe.transform(Xte_te[CATS].astype(str))

    # Scale numerics
    scaler_lr = StandardScaler()
    Xtr_num  = scaler_lr.fit_transform(Xtr_te[lr_num_cols].values.astype('float32'))
    Xval_num = scaler_lr.transform(Xval_te[lr_num_cols].values.astype('float32'))
    Xte_num  = scaler_lr.transform(Xte_te[lr_num_cols].values.astype('float32'))

    # Combine OHE + scaled numerics
    Xtr_final  = sparse.hstack([Xtr_ohe,  sparse.csr_matrix(Xtr_num)])
    Xval_final = sparse.hstack([Xval_ohe, sparse.csr_matrix(Xval_num)])
    Xte_final  = sparse.hstack([Xte_ohe,  sparse.csr_matrix(Xte_num)])

    model_lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
    model_lr.fit(Xtr_final, ytr)
    oof_lr[vali] = model_lr.predict_proba(Xval_final)[:, 1]
    pred_lr += model_lr.predict_proba(Xte_final)[:, 1] / N_SPLITS
    print(f'LR Fold {fold} AUC: {roc_auc_score(yval, oof_lr[vali]):.5f}  '
          f'time={time.time()-fold_start:.0f}s')

print(f'\nLogistic Regression OOF AUC: {roc_auc_score(y_all, oof_lr):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_lr.npy', oof_lr); np.save('pred_lr.npy', pred_lr)

In [ ]:
# ── Hill Climbing Ensemble ────────────────────────────────────
models = {
    'lgb': (oof_lgb,  pred_lgb),
    'xgb': (oof_xgb,  pred_xgb),
    'cat': (oof_cat,   pred_cat),
    'lr':  (oof_lr,    pred_lr),
    'nn':  (oof_nn,    pred_nn),
}

print('Single model OOF AUCs:')
for name, (o, _) in models.items():
    print(f'  {name}: {roc_auc_score(y_all, o):.5f}')

best_name = max(models, key=lambda n: roc_auc_score(y_all, models[n][0]))
best_auc  = roc_auc_score(y_all, models[best_name][0])
print(f'\nStarting from: {best_name} (AUC={best_auc:.5f})')

weights = {best_name: 1.0}
blend_oof  = models[best_name][0].copy().astype(float)
blend_pred = models[best_name][1].copy().astype(float)

TOL = 1e-5
for step in range(30):
    best_step_auc = best_auc
    best_w = None; best_add = None
    for name, (o, _) in models.items():
        for w in np.arange(0.05, 1.0, 0.05):
            trial = (blend_oof + w * o) / (1 + w)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = w; best_add = name
        for w in np.arange(0.05, 0.5, 0.05):
            trial = (blend_oof - w * o) / (1 - w)
            trial = np.clip(trial, 0, 1)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = -w; best_add = name

    if best_add is None or (best_step_auc - best_auc) < TOL:
        print(f'Stopped: improvement {best_step_auc - best_auc:.8f} < TOL={TOL}')
        break

    w = best_w
    blend_oof  = (blend_oof  + w * models[best_add][0]) / (1 + w)
    blend_pred = (blend_pred + w * models[best_add][1]) / (1 + w)
    blend_oof  = np.clip(blend_oof, 0, 1)
    blend_pred = np.clip(blend_pred, 0, 1)
    weights[best_add] = weights.get(best_add, 0) + w
    best_auc = best_step_auc
    print(f'  Step {step}: AUC {best_auc:.6f} | add {best_add} w={w:.3f}')

print(f'\nFinal ensemble OOF AUC: {roc_auc_score(y_all, blend_oof):.5f}')
print('Weights:', weights)

sub['Churn'] = blend_pred
sub.to_csv('submission_ensemble.csv', index=False)
print('Saved submission_ensemble.csv')

In [ ]:
# ── Pseudo-labeling ──────────────────────────────────────────
PSEUDO_THRESH = 0.15
mask = (blend_pred < PSEUDO_THRESH) | (blend_pred > (1 - PSEUDO_THRESH))
print(f'High-confidence test samples: {mask.sum():,} / {len(blend_pred):,} ({mask.mean()*100:.1f}%)')

# For pseudo-labeling, use smoothed TE from full training data
# (no LOO needed here — we're building a final model, not evaluating OOF)
X_all_te = X_all.copy(); Xtest_te = Xtest_raw.copy()
fm = float(y_all.mean())
m_smooth = 200
tmp = X_all_te.assign(_y=y_all.values)
for col in INTER:
    grp = tmp.groupby(col)['_y']
    grp_sum = grp.sum(); grp_count = grp.count()
    grp_mean_smooth = (grp_sum + m_smooth * fm) / (grp_count + m_smooth)
    X_all_te[f'TE_{col}']  = X_all_te[col].map(grp_mean_smooth).fillna(fm).astype('float32')
    Xtest_te[f'TE_{col}']  = Xtest_te[col].map(grp_mean_smooth).fillna(fm).astype('float32')
X_all_te  = X_all_te.drop(columns=INTER)
Xtest_te  = Xtest_te.drop(columns=INTER)

X_pseudo = pd.concat([X_all_te, Xtest_te[mask].reset_index(drop=True)], ignore_index=True)
y_pseudo = np.concatenate([y_all.values.astype('float32'), blend_pred[mask].astype('float32')])
print(f'Expanded training set: {len(X_pseudo):,} rows')

n_est = 500
print(f'Retraining with n_estimators={n_est}...')

pseudo_params = {
    'objective': 'binary', 'n_estimators': n_est,
    'learning_rate': 0.05, 'num_leaves': 127,
    'subsample': 0.8, 'colsample_bytree': 0.6,
    'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_jobs': -1, 'verbose': -1,
}

pred_pseudo = np.zeros(len(Xtest_te), dtype=np.float64)

for seed in LGB_SEEDS:
    Xp = X_pseudo.copy(); Xte = Xtest_te.copy()
    for c in CATS:
        if c in Xp.columns:
            codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
            Xp[c]  = codes[:len(Xp)].astype('int32')
            Xte[c] = codes[len(Xp):].astype('int32')
    for c in Xp.select_dtypes('object').columns:
        codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
        Xp[c]  = codes[:len(Xp)].astype('int32')
        Xte[c] = codes[len(Xp):].astype('int32')
    m = lgb.LGBMRegressor(**{**pseudo_params, 'random_state': seed})
    m.fit(Xp, y_pseudo)
    pred_pseudo += np.clip(m.predict(Xte), 0, 1) / len(LGB_SEEDS)
    print(f'  Seed {seed} done')

PSEUDO_WEIGHT = 0.3
final_pred = (1 - PSEUDO_WEIGHT) * blend_pred + PSEUDO_WEIGHT * pred_pseudo
print(f'\nPseudo blend: {1-PSEUDO_WEIGHT:.0%} ensemble + {PSEUDO_WEIGHT:.0%} pseudo-model')

sub['Churn'] = final_pred
sub.to_csv('submission_final.csv', index=False)
print('Saved submission_final.csv')